In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/application_train.csv')
print(df.shape)
df.head()

In [ ]:
print(df['TARGET'].value_counts())
print(df['TARGET'].value_counts(normalize=True).round(3))

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
print(missing_pct.head(20))

In [ ]:
df.describe().T

In [ ]:
cat_cols = df.select_dtypes('object').columns
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
print(df['CODE_GENDER'].value_counts())

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# Encode categoricals
df_model = df.copy()
cat_cols = df_model.select_dtypes('object').columns
for col in cat_cols:
    df_model[col] = pd.Categorical(df_model[col]).codes

X = df_model.drop(['SK_ID_CURR', 'TARGET'], axis=1)
y = df_model['TARGET']

# 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(n_estimators=1000, random_state=42)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_preds)
    scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
bureau = pd.read_csv('../data/raw/bureau.csv')
print(bureau.shape)
bureau.head()

In [ ]:
# Encode categoricals
bureau_cat = ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']
for col in bureau_cat:
    bureau[col] = pd.Categorical(bureau[col]).codes

# Aggregate to one row per applicant
bureau_agg = bureau.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

# Flatten column names
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

print(bureau_agg.shape)
print(bureau_agg.columns.tolist()[:10])

In [ ]:
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(df.shape)

In [ ]:
previous_application = pd.read_csv('../data/raw/previous_application.csv')
print(previous_application.shape)
previous_application.head()

In [ ]:
previous_application.select_dtypes('object').columns.tolist()

In [ ]:
previous_application_cat = previous_application.select_dtypes('object').columns.tolist()
for col in previous_application_cat:
    previous_application[col] = pd.Categorical(previous_application[col]).codes

previous_application_agg = previous_application.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

previous_application_agg.columns = ['PREV_' + '_'.join(col).upper() for col in previous_application_agg.columns]
previous_application_agg = previous_application_agg.reset_index()

df = df.merge(previous_application_agg, on='SK_ID_CURR', how='left')
print(df.shape)

In [ ]:
# POS_CASH_balance
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
pos_cash_cat = pos_cash.select_dtypes('object').columns.tolist()
for col in pos_cash_cat:
    pos_cash[col] = pd.Categorical(pos_cash[col]).codes
pos_cash_agg = pos_cash.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
pos_cash_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_cash_agg.columns]
pos_cash_agg = pos_cash_agg.reset_index()
df = df.merge(pos_cash_agg, on='SK_ID_CURR', how='left')
print(df.shape)

In [ ]:
# credit_card_balance
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')
credit_card_cat = credit_card.select_dtypes('object').columns.tolist()
for col in credit_card_cat:
    credit_card[col] = pd.Categorical(credit_card[col]).codes
credit_card_agg = credit_card.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
credit_card_agg.columns = ['CC_' + '_'.join(col).upper() for col in credit_card_agg.columns]
credit_card_agg = credit_card_agg.reset_index()
df = df.merge(credit_card_agg, on='SK_ID_CURR', how='left')
print(df.shape)

In [ ]:
# installments_payments
installments = pd.read_csv('../data/raw/installments_payments.csv')
installments_cat = installments.select_dtypes('object').columns.tolist()
for col in installments_cat:
    installments[col] = pd.Categorical(installments[col]).codes
installments_agg = installments.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
installments_agg.columns = ['INST_' + '_'.join(col).upper() for col in installments_agg.columns]
installments_agg = installments_agg.reset_index()
df = df.merge(installments_agg, on='SK_ID_CURR', how='left')
print(df.shape)

In [ ]:
bureau_balance = pd.read_csv('../data/raw/bureau_balance.csv')
print(bureau_balance.shape)
bureau_balance.head()

In [ ]:
# Aggregate bureau_balance by SK_ID_BUREAU
bureau_balance_cat = bureau_balance.select_dtypes('object').columns.tolist()
for col in bureau_balance_cat:
    bureau_balance[col] = pd.Categorical(bureau_balance[col]).codes

bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(['mean', 'max', 'min', 'std', 'count'])
bb_agg.columns = ['BB_' + '_'.join(col).upper() for col in bb_agg.columns]
bb_agg = bb_agg.reset_index()

# Merge into bureau, then aggregate by SK_ID_CURR
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
bureau_full_agg = bureau_full.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
bureau_full_agg.columns = ['BUREAU_BB_' + '_'.join(col).upper() for col in bureau_full_agg.columns]
bureau_full_agg = bureau_full_agg.reset_index()

# Merge into main table
df = df.merge(bureau_full_agg, on='SK_ID_CURR', how='left')
print(df.shape)

In [ ]:
# Encode remaining categoricals in main table
cat_cols = df.select_dtypes('object').columns.tolist()
for col in cat_cols:
    df[col] = pd.Categorical(df[col]).codes

X = df.drop(['SK_ID_CURR', 'TARGET'], axis=1)
y = df['TARGET']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(n_estimators=1000, random_state=42)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_preds)
    scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
import optuna

def objective(trial):
    params = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42
    }
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(-1)])
        
        val_preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, val_preds))
    
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Best AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
best_params = study.best_params
best_params['n_estimators'] = 1000
best_params['random_state'] = 42

# Train on full training data
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y, callbacks=[lgb.log_evaluation(100)])

# Load and prepare test data
test = pd.read_csv('../data/raw/application_test.csv')
test_ids = test['SK_ID_CURR']

In [ ]:
# Encode categoricals
bureau_cat = bureau.select_dtypes('object').columns.tolist()
for col in bureau_cat:
    bureau[col] = pd.Categorical(bureau[col]).codes

# Aggregate to one row per applicant
bureau_agg = bureau.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

# Flatten column names
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

print(bureau_agg.shape)
print(bureau_agg.columns.tolist()[:10])

test = test.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(test.shape)

In [ ]:
previous_application_cat = previous_application.select_dtypes('object').columns.tolist()
for col in previous_application_cat:
    previous_application[col] = pd.Categorical(previous_application[col]).codes

previous_application_agg = previous_application.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

previous_application_agg.columns = ['PREV_' + '_'.join(col).upper() for col in previous_application_agg.columns]
previous_application_agg = previous_application_agg.reset_index()

test = test.merge(previous_application_agg, on='SK_ID_CURR', how='left')
print(test.shape)

In [ ]:
# POS_CASH_balance
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
pos_cash_cat = pos_cash.select_dtypes('object').columns.tolist()
for col in pos_cash_cat:
    pos_cash[col] = pd.Categorical(pos_cash[col]).codes
pos_cash_agg = pos_cash.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
pos_cash_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_cash_agg.columns]
pos_cash_agg = pos_cash_agg.reset_index()
test = test.merge(pos_cash_agg, on='SK_ID_CURR', how='left')
print(test.shape)

In [ ]:
# credit_card_balance
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')
credit_card_cat = credit_card.select_dtypes('object').columns.tolist()
for col in credit_card_cat:
    credit_card[col] = pd.Categorical(credit_card[col]).codes
credit_card_agg = credit_card.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
credit_card_agg.columns = ['CC_' + '_'.join(col).upper() for col in credit_card_agg.columns]
credit_card_agg = credit_card_agg.reset_index()
test = test.merge(credit_card_agg, on='SK_ID_CURR', how='left')
print(test.shape)

In [ ]:
# installments_payments
installments = pd.read_csv('../data/raw/installments_payments.csv')
installments_cat = installments.select_dtypes('object').columns.tolist()
for col in installments_cat:
    installments[col] = pd.Categorical(installments[col]).codes
installments_agg = installments.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
installments_agg.columns = ['INST_' + '_'.join(col).upper() for col in installments_agg.columns]
installments_agg = installments_agg.reset_index()
test = test.merge(installments_agg, on='SK_ID_CURR', how='left')
print(test.shape)

In [ ]:
# Aggregate bureau_balance by SK_ID_BUREAU
bureau_balance_cat = bureau_balance.select_dtypes('object').columns.tolist()
for col in bureau_balance_cat:
    bureau_balance[col] = pd.Categorical(bureau_balance[col]).codes

bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(['mean', 'max', 'min', 'std', 'count'])
bb_agg.columns = ['BB_' + '_'.join(col).upper() for col in bb_agg.columns]
bb_agg = bb_agg.reset_index()

# Merge into bureau, then aggregate by SK_ID_CURR
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
bureau_full_agg = bureau_full.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
bureau_full_agg.columns = ['BUREAU_BB_' + '_'.join(col).upper() for col in bureau_full_agg.columns]
bureau_full_agg = bureau_full_agg.reset_index()

# Merge into main table
test = test.merge(bureau_full_agg, on='SK_ID_CURR', how='left')
print(test.shape)

In [ ]:
# Encode categoricals
cat_cols = test.select_dtypes('object').columns.tolist()
for col in cat_cols:
    test[col] = pd.Categorical(test[col]).codes

# Drop SK_ID_CURR, no TARGET to drop
X_test = test.drop(['SK_ID_CURR'], axis=1)

# Predict
test_preds = final_model.predict_proba(X_test)[:, 1]

# Save submission
submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': test_preds})
submission.to_csv('../outputs/submission.csv', index=False)
print(submission.head())